# SLM-SQL-1.5B: sinh pool BIRD dev bằng vLLM (thiết lập bài báo)

Notebook này chỉ **sinh 16 mẫu/câu** (prompt Appendix E.1, temperature 0.8, `max_new_tokens` 1024) và ghi `pool.jsonl` đúng định dạng `run_sweep`. Judge, CAV và `selection_study` chạy ở máy local trên file tải về.

**Chuẩn bị (một lần):**
1. Trên máy local: `python scripts/pack_kaggle.py --dataset data/benchmarks/bird_dev.json data/benchmarks/bird_dev200.json` → `dist/adaptive-slm-sql-kaggle.zip`.
2. Kaggle → *Datasets* → *New Dataset* → upload file zip, để **Private**.
3. Notebook này: *Add Input* → chọn dataset vừa tạo. *Settings*: **Accelerator = GPU T4 x2** (hoặc P100), **Internet = On** (tải model từ Hugging Face; cần xác minh số điện thoại).
4. Chạy tất cả cell. Nên dùng *Save Version → Save & Run All* để chạy nền, tắt trình duyệt vẫn chạy (tối đa 12 giờ); kết quả nằm ở tab *Output*.

Chạy lại trên cùng thư mục output thì câu đã xong được bỏ qua. **Chạy tiếp từ version trước** (mỗi version là máy mới, `/kaggle/working` trống): *Add Input* → *Notebooks* → chọn chính notebook này (output của version trước); cell sinh pool tự chép `pool.jsonl` cũ sang và chỉ sinh các câu còn thiếu.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

Cài vLLM vào venv Python 3.12 riêng (khoảng 3–5 phút). Image Kaggle hiện chạy Python 3.13, còn vLLM 0.8.5 chỉ có bản cho Python < 3.13. `uv` tự tải Python 3.12; `--exclude-newer` khóa mọi thư viện phụ (transformers, torch…) về thời điểm vLLM 0.8.5 ra mắt để không bị bản mới làm hỏng. Venv nằm ở `/tmp` nên không bị lưu vào *Output*.

T4 không có bfloat16 nên dùng float16; engine V0 chạy được trên T4.

In [ ]:
import glob, os, shutil, subprocess, sys
VENV = "/tmp/vllm-env"
PY = f"{VENV}/bin/python"


def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError(f"Lỗi khi chạy: {cmd}")


if not os.path.exists(PY):
    sh(f"{sys.executable} -m pip install -q uv")
    sh(f"{sys.executable} -m uv venv -q --python 3.12 {VENV}")
    sh(f'{sys.executable} -m uv pip install -q --python {PY} --exclude-newer 2025-05-15 '
       '"vllm==0.8.5.post1" "sqlglot==26.16.4" "PyYAML==6.0.2"')

In [ ]:
r = subprocess.run([PY, "-c", "import sys, vllm; print('python', sys.version.split()[0], '| vllm', vllm.__version__)"],
                   capture_output=True, text=True)
print(r.stdout, r.stderr[-3000:])
assert r.returncode == 0, "Không import được vllm trong venv"

Chép code và database sang `/tmp` (ghi được, không bị lưu vào *Output*). Database phải nằm ở ổ ghi được vì `card_games.sqlite` của BIRD ở chế độ WAL: SQLite cần tạo file `-shm` cạnh nó, nên mở trên `/kaggle/input` (chỉ đọc) sẽ báo `unable to open database file`.

In [ ]:
# Output của version trước (nếu được Add Input) cũng chứa một bản code; chỉ lấy bản có database thật.
src = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/adaptive-slm-sql/config.yaml", recursive=True)
       if os.path.isdir(os.path.join(os.path.dirname(p), "data", "external"))
       and not os.path.islink(os.path.join(os.path.dirname(p), "data", "external"))]
assert src, "Không thấy adaptive-slm-sql/ trong /kaggle/input: kiểm tra Add Input"
src = src[0]
work = "/tmp/adaptive-slm-sql"  # ngoài /kaggle/working để Output chỉ có kết quả
if not os.path.exists(work):
    shutil.copytree(src, work, ignore=shutil.ignore_patterns("external"))
    shutil.copytree(os.path.join(src, "data", "external"), "/tmp/external")
    os.symlink("/tmp/external", os.path.join(work, "data", "external"))
os.chdir(work)
print(os.getcwd(), os.listdir("data/benchmarks"))

## Sinh pool

- `DATASET = bird_dev.json` (1534 câu, so với số của bài báo) hoặc `bird_dev200.json` (200 câu, so với kết quả local).
- **Thêm 48 mẫu** (đủ 64 như bài báo, cấu hình mặc định bên dưới): `POOL = 48`, `SEED_OFFSET = 100000` (mẫu khác hẳn lần 16 mẫu), `GPUS = 2` (T4 x2: mỗi GPU một nửa số câu, ghi vào `shard0/`, `shard1/`). Ở máy local ghép với pool 16 mẫu bằng `scripts/merge_pools.py`.
- **Lần đầu 16 mẫu**: `RUN = "bird-dev-vllm"`, `POOL = 16`, `SEED_OFFSET = 0`, `GPUS = 1`.
- `LIMIT = 20` để thử nhanh trước, rồi đặt `None` (câu đã xong được giữ).
- Tiến độ của từng GPU in mỗi phút; log đầy đủ ở `results/<RUN>/shard*.log`.
- Nếu có `pool.jsonl` từ version trước trong input (xem phần đầu notebook) thì nó được chép sang để chạy tiếp.

In [ ]:
import time

DATASET = "data/benchmarks/bird_dev.json"
RUN = "bird-dev-vllm-extra48"
POOL = 48
SEED_OFFSET = 100000
GPUS = 2
LIMIT = None  # 20 để thử nhanh
OUTPUT = f"/kaggle/working/results/{RUN}"

os.makedirs(OUTPUT, exist_ok=True)
procs = []
for shard in range(GPUS):
    out = f"{OUTPUT}/shard{shard}" if GPUS > 1 else OUTPUT
    name = f"{RUN}/shard{shard}" if GPUS > 1 else RUN
    previous = glob.glob(f"/kaggle/input/**/results/{name}/pool.jsonl", recursive=True)
    if previous and not os.path.exists(f"{out}/pool.jsonl"):
        os.makedirs(out, exist_ok=True)
        shutil.copy(previous[0], f"{out}/pool.jsonl")
        print("Chạy tiếp từ", previous[0])
    cmd = [PY, "-u", "-m", "evaluation.vllm_collect", "--dataset", DATASET, "--output", out,
           "--pool", str(POOL), "--chunk", "32", "--dtype", "half",
           "--seed-offset", str(SEED_OFFSET), "--shard", f"{shard}/{GPUS}"]
    if LIMIT:
        cmd += ["--limit", str(LIMIT)]
    env = dict(os.environ, VLLM_USE_V1="0", CUDA_VISIBLE_DEVICES=str(shard))
    log = f"{OUTPUT}/shard{shard}.log"
    with open(log, "w") as handle:
        procs.append((shard, log, subprocess.Popen(cmd, stdout=handle, stderr=subprocess.STDOUT, env=env)))

seen = [0] * GPUS
while True:
    running = any(proc.poll() is None for _, _, proc in procs)
    for shard, log, _ in procs:
        with open(log, encoding="utf-8", errors="replace") as handle:
            lines = handle.readlines()
        for line in lines[seen[shard]:]:
            if not line.startswith(("INFO", "WARNING", "Capturing", "Loading", "(")) or "Error" in line:
                print(f"[GPU {shard}]", line, end="")
        seen[shard] = len(lines)
    if not running:
        break
    time.sleep(60)
for shard, _, proc in procs:
    print(f"GPU {shard}: exit code", proc.returncode)

EX tạm tính trên phần đã sinh của từng GPU (vote, metric set của BIRD) và oracle:

In [ ]:
for out in sorted({os.path.dirname(p) for p in glob.glob(f"{OUTPUT}/**/pool.jsonl", recursive=True)}):
    print(out)
    !{PY} -m evaluation.progress {out} --dataset {DATASET} | head -n 9

Nén kết quả để tải về (tab *Output* hoặc link bên dưới). Giải nén vào thư mục project local, giữ nguyên `results/<RUN>/` (không lồng `results/results/`).

In [ ]:
archive = shutil.make_archive(f"/kaggle/working/{RUN}", "zip", "/kaggle/working", f"results/{RUN}")
from IPython.display import FileLink
os.chdir("/kaggle/working")
FileLink(os.path.basename(archive))